# code-server on Colab — plug and play

**Runtime → Run all.** No clone, no Drive, no patches to code-server. The only
optional input is an ngrok token; without one it falls back to a Cloudflare quick
tunnel that needs no account.

Nothing here renames processes or spoofs traffic — that layer is not what Colab
enforces on. See `README.md` (written to disk by cell 4) for the layer-by-layer
breakdown. What this does fix: rotating URLs, blank pages, and VMs left idle.

Colab's FAQ does not list remote-IDE use as supported. It works; the runtime is
ephemeral and account-level throttling is always possible.

## Flow, and where the logs are

```
install.sh                     (cell 5, slow: 1–3 min first run)
  apt-get runtime libs  →  code-server tarball  →  ngrok  →  cloudflared

supervise.sh                   (cell 6, returns immediately)
  └─ serve.sh                  (cell 7 watches this live)
       clear stale pidfiles
       port check
       code-server start  →  poll /healthz           (5–90 s)
       tunnel start       →  poll for public URL     (10–60 s)
       write credentials.txt  →  print URL + password
  └─ then: restart on crash, stop when idle
```

Every stage appends to one log:

| File | Contents |
|---|---|
| `/content/.code-server/supervise.out` | **the progress log** — one line per stage |
| `/content/.code-server/code-server.log` | editor server; startup failures land here |
| `/content/.code-server/cloudflared.log` | tunnel registration and the public URL |
| `/content/.code-server/ngrok.log` | JSON; errors like `ERR_NGROK_4018` |
| `/content/.code-server/credentials.txt` | final `url=` / `password=` |

If cell 7 seems stuck, read `supervise.out`: the last line names the stage.

## 1. Environment

In [ ]:
import os, sys, platform, pathlib, shutil

IN_COLAB = "google.colab" in sys.modules or pathlib.Path("/content").is_dir()
BASE = pathlib.Path("/content/colab-cs")

print("colab      :", IN_COLAB)
print("platform   :", platform.platform())
print("machine    :", platform.machine())
print("python     :", platform.python_version())
try:
    print("free disk  :", shutil.disk_usage("/content").free // 2**30, "GiB")
except OSError:
    pass
print("install to :", BASE)

## 2. Configuration

Every value is optional. `TUNNEL` stays `auto` unless you force one:

* `auto` — ngrok if it has credentials, else cloudflared (no account needed)
* `ngrok` — needs `NGROK_TOKEN` (or a Colab Secret of that name); fastest
* `cloudflared` — no account, slightly slower to register, URL changes per run
* `none` — loopback only, for debugging

And the rest of the knobs, all with working defaults:

| Variable | Default | Notes |
|---|---|---|
| `CS_PORT` | auto | pin a port, or leave empty to scan `CS_PORT_RANGE` and take the first free one. The tunnel maps to whatever is chosen |
| `CS_PORT_RANGE` | `3000-5000` | scan range used when `CS_PORT` is empty. Avoids 8080, which `datalab/web/app.js` owns on some Colab runtimes |
| `NGROK_DOMAIN` | — | reserved ngrok domain, so the URL survives restarts |
| `NGROK_AUTH` | — | `user:pass` gate at ngrok's edge, rejects scanners |
| `IDLE_TIMEOUT` | `1800` | shut down after this many seconds with no connections |
| `MAX_RUNTIME` | `39600` | hard stop at 11 h |
| `STATE_ON_DRIVE` | `False` | keep extensions and settings between runtimes |

The resolved port is recorded in `$CS_STATE_DIR/port`, so `stop.sh`, the idle
watchdog and `doctor.sh` always operate on the port actually in use.

In [ ]:
import os

# ── edit here if you want ─────────────────────────────────────────────────────
NGROK_TOKEN    = ""        # or add a Colab Secret named NGROK_TOKEN
TUNNEL         = ""        # "" = auto | "ngrok" | "cloudflared" | "none"
NGROK_DOMAIN   = ""        # reserved domain => stable URL, e.g. "me.ngrok-free.app"
CS_PORT        = ""        # "" = auto: first free port in CS_PORT_RANGE; or pin one, e.g. 3225
CS_PORT_RANGE  = ""        # "" = 3000-5000
NGROK_AUTH     = ""        # optional edge gate "user:pass" (rejects scanners)
IDLE_TIMEOUT   = 1800      # seconds with zero connections before shutdown
MAX_RUNTIME    = 39600     # seconds before forced shutdown (11 h)
STATE_ON_DRIVE = False     # keep extensions and settings between runtimes
CS_EXTENSIONS  = ""        # comma-separated IDs, installed once
# ─────────────────────────────────────────────────────────────────────────────

token = NGROK_TOKEN.strip()
token_source = "config cell" if token else ""
if not token:
    try:
        from google.colab import userdata
        token = (userdata.get("NGROK_TOKEN") or "").strip()
        token_source = "Colab secret" if token else ""
    except Exception:
        pass

os.environ["NGROK_TOKEN"] = token
os.environ["CS_TUNNEL"] = TUNNEL
os.environ["CS_PORT"] = str(CS_PORT)
if CS_PORT_RANGE:
    os.environ["CS_PORT_RANGE"] = CS_PORT_RANGE
os.environ["CS_IDLE_TIMEOUT"] = str(IDLE_TIMEOUT)
os.environ["CS_MAX_RUNTIME"] = str(MAX_RUNTIME)
if NGROK_DOMAIN: os.environ["NGROK_DOMAIN"] = NGROK_DOMAIN
if NGROK_AUTH:   os.environ["NGROK_BASIC_AUTH"] = NGROK_AUTH
if CS_EXTENSIONS: os.environ["CS_EXTENSIONS"] = CS_EXTENSIONS

if STATE_ON_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["CS_STATE_DIR"] = "/content/drive/MyDrive/.code-server"
    os.environ["CS_CACHE_DIR"] = "/content/drive/MyDrive/.code-server/cache"

print("tunnel      :", TUNNEL or "auto")
print("port        :", CS_PORT or f"auto ({CS_PORT_RANGE or '3000-5000'})")
print("ngrok token :", f"yes ({token_source})" if token else "no -> cloudflared fallback")
print("state dir   :", os.environ.get("CS_STATE_DIR", "/content/.code-server"))

## 3. Write the toolkit

The scripts are embedded in the next cell **as plain text**, so you can read and
review exactly what will run — no encoded blobs. The cell just writes them to
`/content/colab-cs` and sets the executable bit; it downloads nothing and
executes nothing.

Prefer the repo? Skip this cell and point `BASE` at a clone:

```bash
git clone --depth 1 https://github.com/recursive-factory/code-server.git /content/cs
```

```python
BASE = pathlib.Path("/content/cs/colab")
```

In [ ]:
import pathlib, stat

# These are the real scripts, verbatim — the same files live in the repo at
# colab/. Written to disk so they can be read, edited and re-run.
FILES = {
    'lib/common.sh': r'''#!/usr/bin/env bash
# colab/lib/common.sh — shared helpers for the Colab scripts.
#
# Source this file, do not execute it:
#   source "$(dirname "$0")/lib/common.sh"
#
# Everything here is deliberately side-effect free apart from installing an ERR
# trap and the colour constants.

if [[ -n "${CS_COMMON_LOADED:-}" ]]; then
  return 0
fi
CS_COMMON_LOADED=1

set -Eeuo pipefail

# shellcheck disable=SC2128  # BASH_SOURCE is the file being sourced
readonly CS_LIB_DIR="$(cd -- "$(dirname -- "${BASH_SOURCE[0]}")" && pwd -P)"
readonly CS_REPO_ROOT="$(cd -- "${CS_LIB_DIR}/../.." && pwd -P)"

# --------------------------------------------------------------- output ----
if [[ -t 1 ]]; then
  CS_C_RESET=$'\033[0m'
  CS_C_BLUE=$'\033[1;34m'
  CS_C_GREEN=$'\033[1;32m'
  CS_C_RED=$'\033[1;31m'
  CS_C_YELLOW=$'\033[1;33m'
  CS_C_DIM=$'\033[2m'
else
  CS_C_RESET='' CS_C_BLUE='' CS_C_GREEN='' CS_C_RED='' CS_C_YELLOW='' CS_C_DIM=''
fi

log()  { printf '%s==>%s %s\n'   "$CS_C_BLUE"   "$CS_C_RESET" "$*"; }
ok()   { printf '%s ok%s %s\n'   "$CS_C_GREEN"  "$CS_C_RESET" "$*"; }
warn() { printf '%swarn%s %s\n'  "$CS_C_YELLOW" "$CS_C_RESET" "$*" >&2; }
die()  { printf '%serr%s %s\n'   "$CS_C_RED"    "$CS_C_RESET" "$*" >&2; exit 1; }
dim()  { printf '%s%s%s\n'       "$CS_C_DIM"    "$*" "$CS_C_RESET"; }

# Report the failing line instead of dying silently. Exempt from the usual
# `&&`/`||`/`if` cases, so it only fires on genuinely unexpected failures.
cs_on_err() {
  local ec=$? line=$1
  printf '%serr%s %s:%s exited with %d\n' \
    "$CS_C_RED" "$CS_C_RESET" "$(basename -- "${BASH_SOURCE[2]:-?}")" "$line" "$ec" >&2
  exit "$ec"
}
trap 'cs_on_err "$LINENO"' ERR

# --------------------------------------------------------------- system ----
cs_is_root() { [[ "$(id -u)" -eq 0 ]]; }

cs_root() {
  if cs_is_root; then "$@"; else sudo -- "$@"; fi
}

cs_need() {
  local cmd=$1 hint=${2:-}
  command -v "$cmd" >/dev/null 2>&1 || die "missing command: ${cmd}${hint:+ (${hint})}"
}

cs_os_field() {
  ( # shellcheck disable=SC1091
    . /etc/os-release 2>/dev/null || exit 0
    printf '%s' "${!1:-}"
  )
}

# Ubuntu 24.04+ renamed libasound2 -> libasound2t64. Colab still ships 22.04,
# but the image changes without notice, so detect instead of hardcoding.
cs_asound_pkg() {
  case "$(cs_os_field VERSION_ID)" in
    24.04*|25.*|26.*|2[7-9].*) printf 'libasound2t64' ;;
    *)                         printf 'libasound2' ;;
  esac
}

cs_arch() {
  case "$(uname -m)" in
    x86_64|amd64)  printf 'amd64' ;;
    aarch64|arm64) printf 'arm64' ;;
    *) die "unsupported architecture: $(uname -m)" ;;
  esac
}

# ------------------------------------------------------------ state dir ----
cs_state_dir() {
  local d="${CS_STATE_DIR:-/content/.code-server}"
  mkdir -p -- "$d" || die "cannot create state dir: ${d}"
  chmod 700 -- "$d" 2>/dev/null || true
  printf '%s' "$d"
}

cs_cache_dir() {
  local d="${CS_CACHE_DIR:-$(cs_state_dir)/cache}"
  mkdir -p -- "$d" || die "cannot create cache dir: ${d}"
  printf '%s' "$d"
}

# ------------------------------------------------------------------ lock ----
CS_LOCK_PATH=''

cs_lock() {
  local name=${1:-lock} sd f
  sd="$(cs_state_dir)"
  f="${sd}/${name}.lock"
  if command -v flock >/dev/null 2>&1; then
    exec 9>"$f" || die "cannot open lock: ${f}"
    flock -n 9 || die "another instance is already running (${f})"
    CS_LOCK_PATH="$f"
  else
    local d="${f}.d"
    mkdir -- "$d" 2>/dev/null || die "another instance is already running (${d})"
    printf '%s\n' "$$" >"${d}/pid"
    CS_LOCK_PATH="$d"
  fi
}

cs_unlock() {
  [[ -n "${CS_LOCK_PATH:-}" ]] || return 0
  if [[ -d "$CS_LOCK_PATH" ]]; then
    rm -rf -- "$CS_LOCK_PATH"
  else
    : >"$CS_LOCK_PATH" 2>/dev/null || true
  fi
  CS_LOCK_PATH=''
  return 0
}

# -------------------------------------------------------------- pidfiles ----
cs_read_pid() { [[ -s "${1:-}" ]] && cat -- "$1" 2>/dev/null || printf ''; }
cs_write_pid() { printf '%s\n' "$2" >"$1"; }

cs_alive() {
  local p
  p="$(cs_read_pid "${1:-}")"
  [[ -n "$p" ]] || return 1
  kill -0 "$p" 2>/dev/null
}

cs_stop_pidfile() {
  local f=$1 label=${2:-process} p i
  p="$(cs_read_pid "$f")"
  rm -f -- "$f"
  [[ -n "$p" ]] || return 0
  kill -0 "$p" 2>/dev/null || return 0
  log "stopping ${label} (pid ${p})"
  kill -TERM "$p" 2>/dev/null || true
  for i in {1..15}; do
    kill -0 "$p" 2>/dev/null || return 0
    sleep 0.3
  done
  kill -KILL "$p" 2>/dev/null || true
  return 0
}

# ------------------------------------------------------------------ http ----
cs_wait_http() {
  local url=$1 timeout=${2:-60} i
  for ((i = 0; i < timeout; i++)); do
    # --noproxy: Colab/Jupyter sometimes exports http_proxy, which would send
    # loopback checks through a proxy that cannot reach 127.0.0.1.
    if curl -fsS --max-time 2 --noproxy '*' "$url" >/dev/null 2>&1; then
      return 0
    fi
    # First start on a fresh VM is not instant; emit a heartbeat so a slow run
    # is distinguishable from a hung one when reading the log afterwards.
    if ((i > 0 && i % 15 == 0)); then
      dim "  waiting for ${url} (${i}s)"
    fi
    sleep 1
  done
  return 1
}

cs_port_in_use() {
  local p=$1
  if command -v ss >/dev/null 2>&1; then
    ss -ltnH 2>/dev/null | awk -v pat=":${p}\$" '$4 ~ pat { found = 1 } END { exit !found }'
  else
    (exec 3<>"/dev/tcp/127.0.0.1/${p}") 2>/dev/null
  fi
}

# Established connections on a local port. Used by the idle watchdog: counting
# sockets is cheap and needs nothing from the editor itself.
cs_active_conns() {
  local p=$1 n
  if command -v ss >/dev/null 2>&1; then
    n="$(ss -Htn state established "sport = :${p}" 2>/dev/null | wc -l | tr -d ' ')"
    printf '%s' "${n:-0}"
    return 0
  fi
  n="$(python3 -c '
import sys
port = int(sys.argv[1])
want = "%04X" % port
n = 0
for f in ("/proc/net/tcp", "/proc/net/tcp6"):
    try:
        rows = open(f).read().splitlines()[1:]
    except OSError:
        continue
    for r in rows:
        c = r.split()
        if len(c) > 3 and c[1].split(":")[1].upper() == want and c[3] == "01":
            n += 1
print(n)
' "$p" 2>/dev/null || true)"
  printf '%s' "${n:-0}"
}

# A code-server process does not keep "code-server --bind-addr" in its argv: the
# bin/code-server wrapper re-execs, so argv looks like
#   .../lib/code-server-<ver>/lib/node .../out/node/entry.js --bind-addr 127.0.0.1:8080
# which is why pattern-matching pkill misses orphaned instances. Resolve the
# socket owner instead.
cs_cmdline() {
  tr '\0' ' ' <"/proc/${1}/cmdline" 2>/dev/null | sed 's/ *$//' || true
}

cs_port_owner_pids() {
  local p=$1
  if command -v ss >/dev/null 2>&1; then
    ss -ltnpH 2>/dev/null | awk -v pat=":${p}\$" '$4 ~ pat' \
      | grep -oE 'pid=[0-9]+' | cut -d= -f2 | sort -u || true
    return 0
  fi
  if command -v lsof >/dev/null 2>&1; then
    lsof -tiTCP:"$p" -sTCP:LISTEN 2>/dev/null | sort -u || true
    return 0
  fi
  if command -v fuser >/dev/null 2>&1; then
    fuser -n tcp "$p" 2>/dev/null | tr -s ' ' '\n' | grep -E '^[0-9]+$' | sort -u || true
  fi
  return 0
}

cs_port_report() {
  local p=$1 owners pid
  owners="$(cs_port_owner_pids "$p")"
  if [[ -z "$owners" ]]; then
    warn "port ${p} is in use but the owner could not be identified"
    return 0
  fi
  for pid in $owners; do
    warn "port ${p} owned by pid ${pid}: $(cs_cmdline "$pid" | cut -c1-100)"
  done
}

# Reclaim the port from a previous run of this stack. Refuses to touch anything
# that does not look like our own processes.
cs_free_port() {
  local p=$1 owners pid cmd i
  owners="$(cs_port_owner_pids "$p")"
  [[ -n "$owners" ]] || return 0

  for pid in $owners; do
    cmd="$(cs_cmdline "$pid")"
    case "$cmd" in
      *entry.js*|*code-server*|*ngrok*|*cloudflared*)
        warn "reclaiming port ${p} from pid ${pid}: ${cmd:0:80}"
        kill -TERM "$pid" 2>/dev/null || true
        ;;
      *)
        warn "port ${p} is held by an unrelated process (pid ${pid}): ${cmd:0:80}"
        return 1
        ;;
    esac
  done

  for i in {1..20}; do
    cs_port_in_use "$p" || return 0
    sleep 0.3
  done
  for pid in $owners; do kill -KILL "$pid" 2>/dev/null || true; done
  for i in {1..10}; do
    cs_port_in_use "$p" || return 0
    sleep 0.3
  done
  return 1
}

# 8080 is not a safe default on Colab: in some runtimes Colab's own frontend
# (datalab/web/app.js) already owns it. An explicit CS_PORT wins; otherwise
# serve.sh picks the first free candidate and records it, and the other scripts
# read that record so they always agree on the port.
cs_port_range() {
  local r="${CS_PORT_RANGE:-3000-5000}" a b
  a="${r%%-*}"
  b="${r##*-}"
  if [[ ! "$a" =~ ^[0-9]+$ || ! "$b" =~ ^[0-9]+$ ]] || ((a >= b)); then
    a=3000
    b=5000
  fi
  printf '%s %s' "$a" "$b"
}

# First port in the range with no listener. One python3 process: a bash loop
# over 2000 ports costs seconds, while connect_ex() on loopback is microseconds
# per port (ECONNREFUSED comes straight back).
cs_find_free_port() {
  local start end found p
  read -r start end <<<"$(cs_port_range)"

  found="$(python3 - "$start" "$end" <<'PYSCAN' 2>/dev/null || true
import socket, sys

start, end = int(sys.argv[1]), int(sys.argv[2])
for port in range(start, end + 1):
    with socket.socket() as s:
        s.settimeout(0.05)
        # non-zero => nothing listening on loopback for that port
        if s.connect_ex(("127.0.0.1", port)) != 0:
            print(port)
            sys.exit(0)
sys.exit(1)
PYSCAN
)"
  if [[ -n "$found" ]]; then
    printf '%s' "$found"
    return 0
  fi

  # Fallback when python3 is missing or the scan failed.
  for ((p = start; p <= end; p++)); do
    if ! (exec 3<>"/dev/tcp/127.0.0.1/${p}") 2>/dev/null; then
      printf '%s' "$p"
      return 0
    fi
  done
  return 1
}

cs_resolved_port() {
  local sd=$1
  if [[ -n "${CS_PORT:-}" ]]; then
    printf '%s' "$CS_PORT"
    return 0
  fi
  if [[ -s "${sd}/port" ]]; then
    head -1 -- "${sd}/port"
    return 0
  fi
  printf '8080'
}

# -------------------------------------------------------------- password ----
# Persisted in the state dir so a restart does not invalidate an open tab.
# CS_PASSWORD overrides without touching disk.
cs_password() {
  local sd=$1 f p
  f="${sd}/password"
  if [[ -n "${CS_PASSWORD:-}" ]]; then
    printf '%s' "$CS_PASSWORD"
    return 0
  fi
  if [[ -s "$f" ]]; then
    cat -- "$f"
    return 0
  fi
  p="$(LC_ALL=C tr -dc 'A-Za-z0-9' </dev/urandom 2>/dev/null | head -c 24 || true)"
  [[ ${#p} -eq 24 ]] || die "failed to generate a password"
  ( umask 077; printf '%s' "$p" >"$f" )
  chmod 600 -- "$f" 2>/dev/null || true
  printf '%s' "$p"
}

# ------------------------------------------------------------------ logs ----
cs_rotate_log() {
  local f=$1 max=${2:-5242880} sz
  [[ -f "$f" ]] || return 0
  sz="$(wc -c <"$f" 2>/dev/null || printf 0)"
  if ((sz > max)); then
    mv -f -- "$f" "${f}.1"
  fi
  return 0
}

cs_dump_log() {
  local f=$1 n=${2:-40}
  [[ -r "$f" ]] || return 0
  warn "last ${n} lines of ${f}:"
  tail -n "$n" -- "$f" >&2 || true
}
''',
    'install.sh': r'''#!/usr/bin/env bash
#
# colab/install.sh — install code-server + a tunnel agent inside a Colab VM.
#
#   bash colab/install.sh
#   WITH_CLOUDFLARED=1 bash colab/install.sh
#   CS_VERSION=4.145.2 bash colab/install.sh     # pin
#   CS_CACHE_DIR=/content/drive/MyDrive/.cs-cache bash colab/install.sh
#
# The release tarball is cached in the state dir. Point CS_CACHE_DIR at Drive
# and a fresh runtime reinstalls in seconds instead of re-downloading ~100 MB.
#
set -Eeuo pipefail
HERE="$(cd -- "$(dirname -- "${BASH_SOURCE[0]}")" && pwd -P)"
# shellcheck source=lib/common.sh
source "${HERE}/lib/common.sh"

CS_PREFIX="${CS_PREFIX:-/usr/local}"
CS_VERSION="${CS_VERSION:-}"
WITH_CLOUDFLARED="${WITH_CLOUDFLARED:-0}"
SKIP_APT="${SKIP_APT:-0}"

# ----------------------------------------------------- resolve the version --
resolve_version() {
  if [[ -n "$CS_VERSION" ]]; then
    printf '%s' "${CS_VERSION#v}"
    return 0
  fi
  local installed
  installed="$(code-server --version 2>/dev/null | awk 'NR==1 { print $1 }' || true)"
  if [[ -n "$installed" ]]; then
    printf '%s' "$installed"
    return 0
  fi
  # HEAD on /releases/latest follows the redirect to /releases/tag/vX.Y.Z and
  # avoids the unauthenticated GitHub API rate limit that shared Colab egress
  # IPs hit constantly.
  local tag
  tag="$(curl -fsSL -o /dev/null -w '%{url_effective}' \
    https://github.com/coder/code-server/releases/latest 2>/dev/null || true)"
  tag="${tag##*/tag/v}"
  if [[ ! "$tag" =~ ^[0-9]+\.[0-9]+\.[0-9]+$ ]]; then
    die "could not resolve the latest code-server version; set CS_VERSION"
  fi
  printf '%s' "$tag"
}

# --------------------------------------------------------------- packages --
install_packages() {
  local pkgs
  pkgs="ca-certificates curl wget xz-utils libnss3 libsecret-1-0 libxkbfile1 libgbm1 libxshmfence1 $(cs_asound_pkg)"
  log "installing runtime libraries: ${pkgs}"
  cs_root apt-get update -qq
  # shellcheck disable=SC2086  # deliberate word splitting of the package list
  DEBIAN_FRONTEND=noninteractive cs_root apt-get install -y -qq --no-install-recommends $pkgs
}

# ------------------------------------------------------------ code-server --
# Official standalone layout, replicated so `code-server` resolves normally:
#   /usr/local/lib/code-server -> code-server-<ver>
#   /usr/local/bin/code-server -> ../lib/code-server/bin/code-server
install_code_server() {
  local version=$1 arch cache tarball root link

  arch="$(cs_arch)"
  cache="$(cs_cache_dir)"
  tarball="${cache}/code-server-${version}-linux-${arch}.tar.gz"
  root="${CS_PREFIX}/lib/code-server-${version}"
  link="${CS_PREFIX}/lib/code-server"

  if [[ -x "${root}/bin/code-server" ]]; then
    ok "code-server ${version} already unpacked at ${root}"
  else
    if [[ ! -s "$tarball" ]]; then
      local url="https://github.com/coder/code-server/releases/download/v${version}/code-server-${version}-linux-${arch}.tar.gz"
      log "downloading ${version} (${arch})"
      if ! curl -fL --retry 3 --retry-delay 2 -# -o "${tarball}.part" "$url"; then
        rm -f -- "${tarball}.part"
        warn "tarball download failed; falling back to the official install script"
        curl -fsSL https://code-server.dev/install.sh \
          | sh -s -- --method=standalone --prefix="$CS_PREFIX" --version="$version"
        return 0
      fi
      mv -f -- "${tarball}.part" "$tarball"
    else
      ok "using cached tarball $(basename -- "$tarball")"
    fi

    log "unpacking into ${CS_PREFIX}/lib"
    local stage extracted
    stage="$(mktemp -d)"
    tar -xzf "$tarball" -C "$stage"
    extracted="$(find "$stage" -maxdepth 1 -mindepth 1 -type d | head -1)"
    if [[ -z "$extracted" ]]; then
      rm -rf -- "$stage"
      die "unexpected tarball layout in $(basename -- "$tarball")"
    fi
    cs_root rm -rf -- "$root"
    cs_root mv -- "$extracted" "$root"
    rm -rf -- "$stage"
  fi

  cs_root ln -sfn -- "code-server-${version}" "$link"
  cs_root mkdir -p -- "${CS_PREFIX}/bin"
  cs_root ln -sfn -- "../lib/code-server/bin/code-server" "${CS_PREFIX}/bin/code-server"
  hash -r

  [[ -x "${CS_PREFIX}/bin/code-server" ]] || die "code-server not on PATH at ${CS_PREFIX}/bin"
  ok "code-server $(code-server --version 2>/dev/null | head -1)"
}

# ------------------------------------------------------------------ ngrok --
install_ngrok() {
  if command -v ngrok >/dev/null 2>&1; then
    ok "ngrok $(ngrok --version 2>/dev/null | head -1)"
    return 0
  fi
  local arch tmp
  arch="$(cs_arch)"
  tmp="$(mktemp -d)"
  log "installing ngrok agent (${arch})"
  curl -fsSL --retry 3 \
    "https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-${arch}.tgz" \
    -o "${tmp}/ngrok.tgz" || die "ngrok download failed"
  tar -xzf "${tmp}/ngrok.tgz" -C "$tmp"
  cs_root install -m 0755 "${tmp}/ngrok" "${CS_PREFIX}/bin/ngrok"
  rm -rf -- "$tmp"
  hash -r
  ok "ngrok $(ngrok --version 2>/dev/null | head -1)"
}

configure_ngrok() {
  if [[ -n "${NGROK_TOKEN:-}" ]]; then
    log "configuring ngrok authtoken"
    ngrok config add-authtoken "$NGROK_TOKEN" >/dev/null 2>&1 \
      || warn "ngrok config add-authtoken failed"
    return 0
  fi
  if ngrok config check >/dev/null 2>&1; then
    ok "ngrok authtoken already configured"
    return 0
  fi
  warn "no NGROK_TOKEN and no authtoken configured — ngrok will fail until you run:"
  warn "  ngrok config add-authtoken <token>    # dashboard.ngrok.com/get-started/your-authtoken"
  return 0
}

# ------------------------------------------------------------ cloudflared --
install_cloudflared() {
  if command -v cloudflared >/dev/null 2>&1; then
    ok "cloudflared $(cloudflared --version 2>/dev/null | awk 'NR==1 { print $3 }')"
    return 0
  fi
  local arch
  arch="$(cs_arch)"
  log "installing cloudflared (${arch})"
  curl -fsSL --retry 3 \
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-${arch}" \
    -o /tmp/cloudflared.bin || die "cloudflared download failed"
  cs_root install -m 0755 /tmp/cloudflared.bin "${CS_PREFIX}/bin/cloudflared"
  rm -f -- /tmp/cloudflared.bin
  hash -r
  ok "cloudflared $(cloudflared --version 2>/dev/null | awk 'NR==1 { print $3 }')"
}

# ------------------------------------------------------------------- main --
main() {
  local sd
  sd="$(cs_state_dir)"

  local version
  version="$(resolve_version)"
  log "target code-server version: ${version}"

  if [[ "$SKIP_APT" != "1" ]]; then
    if command -v code-server >/dev/null 2>&1 && [[ "$(code-server --version 2>/dev/null | awk 'NR==1 { print $1 }')" == "$version" ]]; then
      ok "runtime libraries assumed present (code-server ${version} already installed)"
    else
      install_packages
    fi
  fi

  install_code_server "$version"
  install_ngrok
  configure_ngrok
  if [[ "$WITH_CLOUDFLARED" == "1" ]]; then
    install_cloudflared
  fi

  printf '%s\n' "$version" >"${sd}/.installed-version"

  cat <<EOF

  install complete

  next     bash colab/serve.sh
  or       bash colab/supervise.sh      (restarts + idle/max-runtime watchdog)
  check    bash colab/doctor.sh

EOF
}

main "$@"
''',
    'serve.sh': r'''#!/usr/bin/env bash
#
# colab/serve.sh — start code-server and expose it through a tunnel.
#
#   bash colab/serve.sh                      # ngrok, background, prints URL
#   bash colab/serve.sh --foreground         # block (used by supervise.sh)
#   CS_TUNNEL=cloudflared bash colab/serve.sh
#   NGROK_DOMAIN=my-dev.ngrok-free.app bash colab/serve.sh   # stable hostname
#
# Nothing here modifies code-server. It is a plain Node HTTP server; this script
# only picks flags that suit an ephemeral Colab VM.
#
set -Eeuo pipefail
HERE="$(cd -- "$(dirname -- "${BASH_SOURCE[0]}")" && pwd -P)"
# shellcheck source=lib/common.sh
source "${HERE}/lib/common.sh"

FOREGROUND=0
TUNNEL="${CS_TUNNEL:-auto}"
PORT="${CS_PORT:-}"
AUTH="password"

usage() {
  cat <<'EOF'
usage: serve.sh [options]

  --foreground     remain in the foreground until a child exits
  --tunnel=NAME    auto | ngrok | cloudflared | none (default: $CS_TUNNEL, auto)
  --port=N         local port  (default: first free in $CS_PORT_RANGE, 3000-5000)
  --no-auth        disable authentication (requires CS_ALLOW_INSECURE=1)
  -h, --help
EOF
}

while (($#)); do
  case "$1" in
    --foreground) FOREGROUND=1 ;;
    --tunnel=*)   TUNNEL="${1#*=}" ;;
    --port=*)     PORT="${1#*=}" ;;
    --no-auth)    AUTH="none" ;;
    -h|--help)    usage; exit 0 ;;
    *)            die "unknown argument: $1 (try --help)" ;;
  esac
  shift
done

if [[ -n "$PORT" ]]; then
  [[ "$PORT" =~ ^[0-9]+$ ]] || die "invalid port: ${PORT}"
fi
case "$TUNNEL" in
  auto|ngrok|cloudflared|none) ;;
  *) die "--tunnel must be auto, ngrok, cloudflared or none (got: ${TUNNEL})" ;;
esac

# auto: ngrok if it has credentials, otherwise cloudflared quick tunnel, which
# needs no account at all. That makes a bare notebook run work with zero setup.
if [[ "$TUNNEL" == "auto" ]]; then
  if [[ -n "${NGROK_TOKEN:-}" ]]; then
    TUNNEL=ngrok
  elif command -v ngrok >/dev/null 2>&1 && ngrok config check >/dev/null 2>&1; then
    TUNNEL=ngrok
  elif command -v cloudflared >/dev/null 2>&1; then
    TUNNEL=cloudflared
  else
    TUNNEL=ngrok
  fi
  log "auto-selected tunnel: ${TUNNEL}"
fi

if [[ "$AUTH" == "none" ]]; then
  [[ "${CS_ALLOW_INSECURE:-0}" == "1" ]] \
    || die "--no-auth requires CS_ALLOW_INSECURE=1"
  [[ "$TUNNEL" == "none" ]] \
    || die "--no-auth cannot be combined with a public tunnel"
fi

cs_need code-server "run: bash colab/install.sh"
cs_need curl
cs_need python3

SD="$(cs_state_dir)"
cs_lock serve

# Resolve the port now that the state dir exists, and record it so stop.sh,
# supervise.sh and doctor.sh operate on the same one.
if [[ -z "$PORT" ]]; then
  read -r _pstart _pend <<<"$(cs_port_range)"
  PORT="$(cs_find_free_port)" \
    || die "no free port in ${_pstart}-${_pend}; widen CS_PORT_RANGE or pass --port=<n>"
  log "auto-selected port ${PORT} (first free in ${_pstart}-${_pend})"
fi
printf '%s\n' "$PORT" >"${SD}/port"

CS_PID=''
TUN_PID=''
PUBLIC_URL=''

cleanup_children() {
  [[ -n "$CS_PID" ]] && cs_stop_pidfile "${SD}/cs.pid" code-server
  [[ -n "$TUN_PID" ]] && cs_stop_pidfile "${SD}/${TUNNEL}.pid" "$TUNNEL"
  return 0
}

on_exit() {
  if ((FOREGROUND)); then cleanup_children; fi
  cs_unlock
}
trap on_exit EXIT
# Turn signals into a normal exit so the EXIT trap tears the children down.
trap 'exit 143' TERM INT HUP

# ------------------------------------------------------- stale instances ----
log "clearing stale instances"
cs_stop_pidfile "${SD}/cs.pid" code-server
cs_stop_pidfile "${SD}/ngrok.pid" ngrok
cs_stop_pidfile "${SD}/cloudflared.pid" cloudflared

if cs_port_in_use "$PORT"; then
  log "port ${PORT} is occupied; attempting to reclaim it"
  if ! cs_free_port "$PORT"; then
    cs_port_report "$PORT"
    printf '%serr%s port %s is busy and not reclaimable — free it or use --port=<n>\n' \
      "$CS_C_RED" "$CS_C_RESET" "$PORT" >&2
    # Exit code 2 means "retrying will not help"; supervise.sh stops on it.
    exit 2
  fi
  ok "port ${PORT} reclaimed"
fi

# --------------------------------------------------------- extensions ----
# Installed before the server starts so the marker file and the extension dir
# cannot race with the extension host.
install_extensions() {
  local sd=$1 marker ext
  marker="${sd}/.extensions-$(printf '%s' "$CS_EXTENSIONS" | cksum | cut -d' ' -f1)"
  if [[ -f "$marker" ]]; then
    ok "extensions already installed"
    return 0
  fi
  local IFS=','
  for ext in $CS_EXTENSIONS; do
    [[ -n "$ext" ]] || continue
    log "installing extension ${ext}"
    if code-server \
        --user-data-dir "${sd}/data" \
        --extensions-dir "${sd}/extensions" \
        --install-extension "$ext" --force \
        >>"${sd}/extensions.log" 2>&1; then
      ok "extension ${ext}"
    else
      warn "extension ${ext} failed (see ${sd}/extensions.log)"
    fi
  done
  : >"$marker"
}
if [[ -n "${CS_EXTENSIONS:-}" ]]; then
  install_extensions "$SD"
fi

# -------------------------------------------------------- code-server ----
CODE_PASSWORD="$(cs_password "$SD")"

cs_args=(
  --bind-addr "127.0.0.1:${PORT}"
  --auth "$AUTH"
  --user-data-dir "${SD}/data"
  --extensions-dir "${SD}/extensions"
  --disable-telemetry
  --disable-update-check
  --disable-workspace-trust
)

log "starting code-server on 127.0.0.1:${PORT}"
cs_rotate_log "${SD}/code-server.log"
PASSWORD="$CODE_PASSWORD" nohup code-server "${cs_args[@]}" \
  >"${SD}/code-server.log" 2>&1 9>&- &
CS_PID=$!
cs_write_pid "${SD}/cs.pid" "$CS_PID"

if ! cs_wait_http "http://127.0.0.1:${PORT}/healthz" "${CS_HEALTH_TIMEOUT:-90}"; then
  cs_dump_log "${SD}/code-server.log" 60
  cs_stop_pidfile "${SD}/cs.pid" code-server
  CS_PID=''
  die "code-server did not become healthy"
fi
ok "code-server healthy (pid ${CS_PID})"

# ------------------------------------------------------------- tunnels ----
ngrok_public_url() {
  curl -fsS --max-time 2 --noproxy '*' "http://127.0.0.1:4040/api/tunnels" 2>/dev/null \
    | python3 -c '
import json, sys
try:
    data = json.load(sys.stdin)
except Exception:
    sys.exit(0)
for t in data.get("tunnels", []):
    url = t.get("public_url", "")
    if url.startswith("https://"):
        print(url)
        break
' 2>/dev/null || true
}

start_ngrok() {
  cs_need ngrok "run: bash colab/install.sh"

  if [[ -n "${NGROK_TOKEN:-}" ]]; then
    ngrok config add-authtoken "$NGROK_TOKEN" >/dev/null 2>&1 \
      || warn "ngrok authtoken refresh failed"
  fi

  local args=(http)
  if [[ -n "${NGROK_DOMAIN:-}" ]]; then
    args+=("--url=https://${NGROK_DOMAIN}")
  fi
  if [[ -n "${NGROK_BASIC_AUTH:-}" ]]; then
    # Extra gate at ngrok's edge: rejects scanners before they ever reach
    # code-server. Free tier hostnames get probed within minutes.
    args+=("--basic-auth=${NGROK_BASIC_AUTH}")
  fi
  args+=("$PORT" --log=stdout --log-format=json)

  log "opening ngrok tunnel"
  cs_rotate_log "${SD}/ngrok.log"
  nohup ngrok "${args[@]}" >"${SD}/ngrok.log" 2>&1 9>&- &
  TUN_PID=$!
  cs_write_pid "${SD}/ngrok.pid" "$TUN_PID"

  local i
  for ((i = 0; i < 45; i++)); do
    PUBLIC_URL="$(ngrok_public_url)"
    [[ -n "$PUBLIC_URL" ]] && return 0
    kill -0 "$TUN_PID" 2>/dev/null || break
    if ((i > 0 && i % 10 == 0)); then dim "  waiting for ngrok URL (${i}s)"; fi
    sleep 1
  done

  cs_dump_log "${SD}/ngrok.log" 40
  warn "no public URL from ngrok"
  warn "common causes: missing authtoken (ngrok config add-authtoken <t>),"
  warn "free-tier session limit (one agent at a time), or a reserved-domain mismatch"
  return 1
}

start_cloudflared() {
  cs_need cloudflared "run: WITH_CLOUDFLARED=1 bash colab/install.sh"

  local args=(tunnel)
  if [[ -n "${CF_TOKEN:-}" ]]; then
    # Named tunnel: stable hostname on a domain you control, no interstitial
    # page, survives restarts. This is the answer to "the ngrok URL keeps
    # changing" — proper hosting rather than a disguise.
    args+=(run --token "$CF_TOKEN")
  else
    args+=(--url "http://127.0.0.1:${PORT}")
  fi
  args+=(--no-autoupdate)

  log "opening cloudflared tunnel"
  cs_rotate_log "${SD}/cloudflared.log"
  nohup cloudflared "${args[@]}" >"${SD}/cloudflared.log" 2>&1 9>&- &
  TUN_PID=$!
  cs_write_pid "${SD}/cloudflared.pid" "$TUN_PID"

  if [[ -n "${CF_TUNNEL_URL:-}" ]]; then
    PUBLIC_URL="$CF_TUNNEL_URL"
    return 0
  fi

  local i
  for ((i = 0; i < 45; i++)); do
    PUBLIC_URL="$(grep -oE 'https://[a-z0-9-]+\.trycloudflare\.com' \
      "${SD}/cloudflared.log" 2>/dev/null | head -1 || true)"
    [[ -n "$PUBLIC_URL" ]] && return 0
    kill -0 "$TUN_PID" 2>/dev/null || break
    if ((i > 0 && i % 10 == 0)); then dim "  waiting for cloudflared URL (${i}s)"; fi
    sleep 1
  done

  cs_dump_log "${SD}/cloudflared.log" 40
  warn "no public URL from cloudflared"
  warn "with CF_TOKEN set, pass CF_TUNNEL_URL=https://your.hostname so it can be reported"
  return 1
}

case "$TUNNEL" in
  ngrok)       start_ngrok       || die "tunnel startup failed" ;;
  cloudflared) start_cloudflared || die "tunnel startup failed" ;;
  none)        PUBLIC_URL="http://127.0.0.1:${PORT}" ;;
esac

# ------------------------------------------------------------- report ----
{
  printf 'url=%s\n' "$PUBLIC_URL"
  printf 'password=%s\n' "$CODE_PASSWORD"
  printf 'tunnel=%s\n' "$TUNNEL"
  printf 'port=%s\n' "$PORT"
  printf 'state=%s\n' "$SD"
} >"${SD}/credentials.txt"
chmod 600 -- "${SD}/credentials.txt" 2>/dev/null || true

printf '\n'
dim  '  ────────────────────────────────────────────────────────────'
printf '   %s  %s\n' "$(printf '\033[1mcode-server\033[0m')" "$PUBLIC_URL"
printf '   password     %s\n' "$CODE_PASSWORD"
[[ -n "${NGROK_BASIC_AUTH:-}" ]] && printf '   basic auth   %s\n' "$NGROK_BASIC_AUTH"
dim  '  ────────────────────────────────────────────────────────────'
printf '\n'
dim "  state   ${SD}"
dim "  logs    ${SD}/code-server.log  ${SD}/${TUNNEL}.log"
dim "  stop    bash colab/stop.sh"
printf '\n'

if [[ "$TUNNEL" == "ngrok" && -z "${NGROK_DOMAIN:-}" ]]; then
  warn "ngrok free tier shows a warning page on the first browser hit."
  warn "Click through it, or the editor's JS assets come back as HTML (blank page)."
  warn "Set NGROK_DOMAIN=<reserved-domain> to keep one hostname across restarts,"
  warn "or CS_TUNNEL=cloudflared for no interstitial at all."
fi

# --------------------------------------------------------- foreground ----
if ((FOREGROUND)); then
  log "foreground mode; Ctrl-C to stop"
  while :; do
    if ! kill -0 "$CS_PID" 2>/dev/null; then
      warn "code-server exited"
      exit 1
    fi
    if [[ -n "$TUN_PID" ]] && ! kill -0 "$TUN_PID" 2>/dev/null; then
      warn "tunnel exited"
      exit 1
    fi
    sleep 2
  done
fi
''',
    'supervise.sh': r'''#!/usr/bin/env bash
#
# colab/supervise.sh — keep the stack alive and stop it before Colab does.
#
#   bash colab/supervise.sh
#   CS_IDLE_TIMEOUT=900 CS_MAX_RUNTIME=21600 bash colab/supervise.sh
#
# Two jobs:
#   1. restart serve.sh with exponential backoff if code-server or the tunnel dies
#   2. shut the stack down when nobody is connected, or when the runtime has been
#      held for too long
#
# (2) is the honest half of "less noticeable": a GPU VM held open for hours with
# no interactive traffic is the pattern that gets accounts throttled. Releasing
# it when idle is both better manners and fewer sessions to be flagged.
#
set -Eeuo pipefail
HERE="$(cd -- "$(dirname -- "${BASH_SOURCE[0]}")" && pwd -P)"
# shellcheck source=lib/common.sh
source "${HERE}/lib/common.sh"

PORT="${CS_PORT:-}"
IDLE_TIMEOUT="${CS_IDLE_TIMEOUT:-1800}"     # 30 min with zero connections
MAX_RUNTIME="${CS_MAX_RUNTIME:-39600}"      # 11 h, under Colab's hard cap
HEALTHY_AFTER="${CS_HEALTHY_AFTER:-600}"    # uptime that resets the backoff
MAX_BACKOFF="${CS_MAX_BACKOFF:-60}"
MAX_RESTARTS="${CS_MAX_RESTARTS:-20}"

[[ "$IDLE_TIMEOUT" =~ ^[0-9]+$ ]] || die "CS_IDLE_TIMEOUT must be a number of seconds (0 disables)"
[[ "$MAX_RUNTIME" =~ ^[0-9]+$ ]]  || die "CS_MAX_RUNTIME must be a number of seconds"

cs_need code-server "run: bash colab/install.sh"
cs_need curl

SD="$(cs_state_dir)"
PORT="${PORT:-$(cs_resolved_port "$SD")}"
cs_lock supervise
trap 'cs_unlock' EXIT

stack_alive() {
  local cs_pid
  cs_pid="$(cs_read_pid "${SD}/cs.pid")"
  if [[ -z "$cs_pid" ]] || ! kill -0 "$cs_pid" 2>/dev/null; then
    return 1
  fi
  # The tunnel counts too. If cloudflared/ngrok dies while code-server lives on,
  # the public hostname stays up at Cloudflare's edge but has no connector
  # behind it, so visitors get error 1033. Treat that as a dead stack so the
  # supervisor restarts and publishes a fresh URL, instead of serving a URL that
  # can never work again.
  if [[ -s "${SD}/ngrok.pid" || -s "${SD}/cloudflared.pid" ]]; then
    cs_alive "${SD}/ngrok.pid" || cs_alive "${SD}/cloudflared.pid" || return 1
  fi
  return 0
}

# Returns: 0 = still running, 10 = idle shutdown, 11 = max runtime reached,
#          1 = something died (caller restarts)
watch_stack() {
  local started=$SECONDS last_active=$SECONDS conns
  while stack_alive; do
    sleep 5
    conns="$(cs_active_conns "$PORT")"
    if [[ "${conns:-0}" -gt 0 ]]; then
      last_active=$SECONDS
    fi
    # CS_IDLE_TIMEOUT=0 disables the idle watchdog entirely.
    if ((IDLE_TIMEOUT > 0 && SECONDS - last_active > IDLE_TIMEOUT)); then
      log "no connections for ${IDLE_TIMEOUT}s — shutting down"
      CS_FROM_SUPERVISOR=1 bash "${HERE}/stop.sh" || true
      return 10
    fi
    if ((SECONDS - started > MAX_RUNTIME)); then
      log "max runtime ${MAX_RUNTIME}s reached — shutting down"
      CS_FROM_SUPERVISOR=1 bash "${HERE}/stop.sh" || true
      return 11
    fi
  done
  return 1
}

restarts=0
delay=2

while :; do
  t0=$SECONDS

  srv_rc=0
  bash "${HERE}/serve.sh" "$@" 9>&- || srv_rc=$?

  if ((srv_rc == 0)); then
    PORT="$(cs_resolved_port "$SD")"
    rc=0
    watch_stack || rc=$?
    if ((rc == 1)); then
      warn "stack died (code-server or tunnel exited) — restarting; the public URL will change"
    fi
    case "$rc" in
      10|11) log "stopped cleanly"; exit 0 ;;
    esac
    [[ "$rc" -eq 0 ]] && continue
  elif ((srv_rc == 2)); then
    # Unrecoverable (e.g. the port is held by something that is not ours).
    # Restarting would just burn the retry budget and hide the real message.
    die "cannot start on port ${PORT}; see the error above"
  else
    warn "serve.sh failed to start (exit ${srv_rc})"
  fi

  uptime=$((SECONDS - t0))
  if ((uptime >= HEALTHY_AFTER)); then
    delay=2
    restarts=0
  else
    restarts=$((restarts + 1))
    delay=$((delay * 2))
    ((delay > MAX_BACKOFF)) && delay=$MAX_BACKOFF
  fi

  if ((restarts > MAX_RESTARTS)); then
    cs_dump_log "${SD}/code-server.log" 40
    die "${restarts} failed restarts in a row; giving up"
  fi

  warn "restarting in ${delay}s (uptime ${uptime}s, attempt ${restarts})"
  sleep "$delay"
done
''',
    'stop.sh': r'''#!/usr/bin/env bash
#
# colab/stop.sh — stop code-server and any tunnel started by serve.sh.
#
#   bash colab/stop.sh
#
set -Eeuo pipefail
HERE="$(cd -- "$(dirname -- "${BASH_SOURCE[0]}")" && pwd -P)"
# shellcheck source=lib/common.sh
source "${HERE}/lib/common.sh"

SD="$(cs_state_dir)"
PORT="$(cs_resolved_port "$SD")"

# The supervisor must go first: it restarts serve.sh, so killing the children
# while it still runs just gets them relaunched a few seconds later.
sup_pid="$(cs_read_pid "${SD}/supervise.pid")"
if [[ -n "$sup_pid" ]] && kill -0 "$sup_pid" 2>/dev/null; then
  log "stopping supervisor (pid ${sup_pid})"
  # It was launched with start_new_session, so its pid is also its process group.
  kill -TERM -- "-${sup_pid}" 2>/dev/null || kill -TERM "$sup_pid" 2>/dev/null || true
  for _ in {1..15}; do
    kill -0 "$sup_pid" 2>/dev/null || break
    sleep 0.3
  done
  kill -KILL -- "-${sup_pid}" 2>/dev/null || true
elif [[ "${CS_FROM_SUPERVISOR:-0}" != "1" ]]; then
  # No pidfile (older run, or it was lost) — match the supervisor's own argv.
  pkill -f 'supervise\.sh$' 2>/dev/null || true
fi
rm -f -- "${SD}/supervise.pid"

cs_stop_pidfile "${SD}/cs.pid" code-server
cs_stop_pidfile "${SD}/ngrok.pid" ngrok
cs_stop_pidfile "${SD}/cloudflared.pid" cloudflared

# Fallback for a run whose pidfile was lost. The patterns are anchored to the
# full argv on purpose: this repository's directory is itself named
# "code-server", so a bare `pkill -f code-server` would also match — and kill —
# the shell that invoked this script.
# Matches the real argv of a code-server process (the wrapper re-execs to
# "node .../out/node/entry.js --bind-addr ..."), not just the wrapper name.
pkill -f 'entry\.js --bind-addr' 2>/dev/null || true
pkill -f 'code-server --bind-addr' 2>/dev/null || true
pkill -f 'ngrok http' 2>/dev/null || true
pkill -f 'cloudflared tunnel' 2>/dev/null || true

rm -f -- "${SD}/cs.pid" "${SD}/ngrok.pid" "${SD}/cloudflared.pid"

# Last resort: reclaim the port from an orphan that no pidfile knows about.
if cs_port_in_use "$PORT"; then
  cs_free_port "$PORT" || cs_port_report "$PORT"
fi

ok "stopped"
''',
    'doctor.sh': r'''#!/usr/bin/env bash
#
# colab/doctor.sh — report the state of the Colab environment and the stack.
#
#   bash colab/doctor.sh
#
# Read-only: changes nothing. Run it before filing a "it doesn't work" note.
#
set -Eeuo pipefail
HERE="$(cd -- "$(dirname -- "${BASH_SOURCE[0]}")" && pwd -P)"
# shellcheck source=lib/common.sh
source "${HERE}/lib/common.sh"

SD="$(cs_state_dir)"
PORT="$(cs_resolved_port "$SD")"
problems=0

bad() { warn "$*"; problems=$((problems + 1)); }

kv() { printf '   %-22s %s\n' "$1" "$2"; }

section() { printf '\n%s\n' "$(printf '\033[1m%s\033[0m' "$*")"; }

section "host"
kv "architecture" "$(uname -m)"
kv "os" "$(cs_os_field PRETTY_NAME)"
kv "kernel" "$(uname -r)"
kv "user" "$(id -un) (uid $(id -u))"
kv "python" "$(command -v python3 >/dev/null 2>&1 && python3 -V 2>&1 || echo missing)"
kv "bash" "$BASH_VERSION"

section "resources"
mem_gb="$(awk '/MemTotal/ { printf "%.1f", $2 / 1048576 }' /proc/meminfo 2>/dev/null || echo '?')"
kv "memory" "${mem_gb} GB"
if [[ -d /content ]]; then
  kv "disk /content" "$(df -h /content 2>/dev/null | awk 'NR==2 { print $4 " free of " $2 }')"
fi
kv "state dir" "${SD}"
kv "state size" "$(du -sh "$SD" 2>/dev/null | awk '{ print $1 }' || echo '0')"

section "code-server"
if command -v code-server >/dev/null 2>&1; then
  kv "path" "$(command -v code-server)"
  kv "version" "$(code-server --version 2>/dev/null | head -1 || echo '?')"
  [[ -f "${SD}/.installed-version" ]] && kv "installed tag" "$(cat "${SD}/.installed-version")"
else
  bad "code-server is not on PATH (run: bash colab/install.sh)"
fi

section "tunnels"
if command -v ngrok >/dev/null 2>&1; then
  kv "ngrok" "$(ngrok --version 2>/dev/null | head -1)"
  if ngrok config check >/dev/null 2>&1; then
    kv "ngrok authtoken" "configured"
  else
    bad "ngrok authtoken not configured (ngrok config add-authtoken <token>)"
  fi
else
  kv "ngrok" "not installed"
fi
if command -v cloudflared >/dev/null 2>&1; then
  kv "cloudflared" "$(cloudflared --version 2>/dev/null | awk 'NR==1 { print $3 }')"
else
  kv "cloudflared" "not installed"
fi

section "runtime state"
if cs_alive "${SD}/cs.pid"; then
  kv "code-server" "running (pid $(cs_read_pid "${SD}/cs.pid"))"
else
  kv "code-server" "not running"
fi
if cs_alive "${SD}/ngrok.pid"; then
  kv "ngrok" "running (pid $(cs_read_pid "${SD}/ngrok.pid"))"
fi
if cs_alive "${SD}/cloudflared.pid"; then
  kv "cloudflared" "running (pid $(cs_read_pid "${SD}/cloudflared.pid"))"
fi

kv "port ${PORT}" "$(cs_port_in_use "$PORT" && echo 'in use' || echo 'free')"
owners="$(cs_port_owner_pids "$PORT")"
for pid in $owners; do
  kv "port ${PORT} owner" "pid ${pid}: $(cs_cmdline "$pid" | cut -c1-70)"
done
kv "established conns" "$(cs_active_conns "$PORT")"

if curl -fsS --max-time 2 --noproxy '*' "http://127.0.0.1:${PORT}/healthz" >/dev/null 2>&1; then
  kv "healthz" "ok"
else
  kv "healthz" "not responding"
fi

if [[ -s "${SD}/credentials.txt" ]]; then
  section "credentials (from the last serve.sh run)"
  # Password intentionally not printed here; read the 0600 file if you need it.
  sed -e 's/^password=.*/password=<hidden; see the file>/' "${SD}/credentials.txt" \
    | sed 's/^/   /'
fi

section "public URL"
if [[ -s "${SD}/credentials.txt" ]]; then
  pub_url="$(sed -n 's/^url=//p' "${SD}/credentials.txt" | head -1)"
  pub_tunnel="$(sed -n 's/^tunnel=//p' "${SD}/credentials.txt" | head -1)"
  kv "tunnel" "${pub_tunnel:-?}"
  kv "url" "${pub_url:-?}"
  case "$pub_url" in
    https://*)
      resp="$(curl -sS --max-time 8 --noproxy '*' "$pub_url" 2>/dev/null || true)"
      status="$(curl -s -o /dev/null -w '%{http_code}' --max-time 8 --noproxy '*' "$pub_url" 2>/dev/null || true)"
      case "$resp" in
        *"error code: 1033"*)
          bad "public URL is dead (Cloudflare 1033): no connector attached — the tunnel process is gone" ;;
        *"error code: 502"*|*"error code: 530"*)
          bad "public URL returns Cloudflare ${status}: tunnel up but origin not reachable" ;;
        *)
          kv "reachable" "HTTP ${status:-?}" ;;
      esac
      ;;
  esac
else
  kv "url" "no credentials.txt yet (stack has not started in this state dir)"
fi

section "logs"
for f in code-server.log ngrok.log cloudflared.log extensions.log; do
  if [[ -f "${SD}/${f}" ]]; then
    printf '   %-22s %s bytes\n' "$f" "$(wc -c <"${SD}/${f}" | tr -d ' ')"
  fi
done

printf '\n'
if ((problems == 0)); then
  ok "no problems detected"
else
  warn "${problems} problem(s) detected"
fi
''',
    'README.md': r'''# code-server on Google Colab

Runs code-server in a Colab VM and exposes it over a tunnel.

**No patches to code-server.** It is a plain Node HTTP server; nothing in `src/`
or `patches/` needs to change. These scripts only pick flags appropriate to an
ephemeral Colab VM, install the runtime libraries, and manage the tunnel.

```
colab/
├── install.sh       # libraries + code-server tarball (cached) + tunnel agent
├── serve.sh         # start code-server, wait for health, open tunnel, report
├── supervise.sh     # restart with backoff + idle / max-runtime watchdog
├── stop.sh          # stop everything
├── doctor.sh        # read-only diagnostics
├── lib/common.sh    # shared helpers (logging, locks, pidfiles, probes)
├── code-server-min.ipynb    # 14 cells, ~110 lines: fetches the scripts
└── code-server.ipynb        # 19 cells, ~1650 lines: scripts embedded inline
```

## What a "trace" actually is, and what a wrapper can do about it

Worth reading before asking for process renaming, because the layers are not
equally reachable:

| Layer | Observed by | Changeable by a wrapper? |
|---|---|---|
| `/proc/<pid>/cmdline`, process name | anything inside the VM | yes, trivially — and it does not matter: Colab's enforcement is not a per-process scanner. code-server's own children (`node`, extension hosts, every terminal you open) keep the real names, so a renamed parent is *inconsistent*, which is a worse signal than an honest one. |
| filesystem writes | the VM itself | yes, pointless — the disk is wiped per runtime |
| egress SNI / hostname (`*.ngrok-free.app`) | the network path | only by using your own domain on a named tunnel. That is not disguise, it is hosting — see below. |
| tunnel-side HTTP/WS shape (`/websocket`, asset paths) | ngrok / Cloudflare edge, not Google | no reason to care |
| runtime behaviour: idle GPU VM, session length, zero executed cells, memory/CPU profile | Google, account level | **no** — and this is the layer that gets actioned |
| account / session count / IP reputation | Google | no |

So the two layers a wrapper *can* touch are the two nobody enforces on, and the
one that matters is untouched by any script. What this tooling does instead is
address the underlying complaints properly:

* **rotating URL** → named Cloudflare tunnel on your own domain, stable across
  restarts (`CF_TOKEN` + `CF_TUNNEL_URL`), no interstitial page
* **blank page on ngrok** → reserved ngrok domain (`NGROK_DOMAIN`)
* **VM held open for hours for nothing** → `supervise.sh` idle and max-runtime
  watchdogs. This is also the honest fix for "less noticeable": a GPU VM idling
  with no interactive traffic is the actual pattern that gets throttled, and
  releasing it costs nothing.

Traffic or process respoofing is deliberately not implemented.

## Quick start

```bash
export NGROK_TOKEN=...        # dashboard.ngrok.com/get-started/your-authtoken
bash colab/install.sh
bash colab/serve.sh
```

## Which notebook

| | `code-server-min.ipynb` | `code-server.ipynb` |
|---|---|---|
| size | 14 cells, ~110 lines | 19 cells, ~1650 lines |
| scripts | fetched from this repo | embedded as plain text |
| needs network | yes, to fetch | no |
| edits the scripts locally | yes, in the notebook | yes, in the notebook |
| regenerated by | nothing, edit directly | `build-notebook.py` |

Use the minimal one unless you need an offline notebook or a single
self-contained file to hand to someone.

## Flow and logs

```
install.sh                   slow: 1–3 min on a fresh runtime
  apt runtime libs → code-server tarball (~100 MB) → ngrok → cloudflared

supervise.sh                 returns immediately, runs detached
  └─ serve.sh
       clear stale pidfiles
       port check
       code-server → poll /healthz            5–90 s
       tunnel      → poll for public URL     10–60 s
       write credentials.txt → print URL + password
  └─ restart on crash, stop on idle / max runtime
```

| File | Contents |
|---|---|
| `$CS_STATE_DIR/supervise.out` | **progress log**, one line per stage |
| `$CS_STATE_DIR/code-server.log` | editor server; startup crashes land here |
| `$CS_STATE_DIR/cloudflared.log` | tunnel registration, public URL |
| `$CS_STATE_DIR/ngrok.log` | JSON; look for `ERR_NGROK_4018`, `ERR_NGROK_108` |
| `$CS_STATE_DIR/credentials.txt` | `url=` / `password=` / `tunnel=` |

Health and tunnel waits emit a heartbeat every 10–15 s (`waiting for ... (30s)`),
so a slow start is distinguishable from a hung one. If `serve.sh` stops making
progress, the last line of `supervise.out` names the stage.

First run is slow for reasons that are not code-server: `apt-get` on a fresh
image, ~150 MB of downloads, and tunnel registration. `auto` picks ngrok when it
has a token (fast) and cloudflared otherwise (no account, slower to register).

## Tunnels

### ngrok (default)

```bash
bash colab/serve.sh                                      # auto-selects a tunnel
CS_TUNNEL=ngrok bash colab/serve.sh
NGROK_DOMAIN=my-dev.ngrok-free.app bash colab/serve.sh   # reserved, stable
NGROK_BASIC_AUTH='user:pass' bash colab/serve.sh         # gate at the edge
```

Free tier shows a "You are about to visit" page on the first browser hit. Click
through it before the editor loads, otherwise the JS assets are served as HTML
and you get a blank page. Free tier also allows one agent session and rotates
the hostname unless you reserve a domain.

### cloudflared (no interstitial, supports a real domain)

```bash
WITH_CLOUDFLARED=1 bash colab/install.sh
bash colab/serve.sh --tunnel=cloudflared            # https://<rand>.trycloudflare.com
```

Named tunnel, stable hostname, no warning page:

```bash
cloudflared tunnel login
cloudflared tunnel create colab
cloudflared tunnel route dns colab code.example.com
# then, per session:
CF_TOKEN=<tunnel-token> CF_TUNNEL_URL=https://code.example.com \
  bash colab/serve.sh --tunnel=cloudflared
```

This is the durable option. Nothing is being hidden: the traffic is a normal
TLS connection to a domain you control.

### Not an option: Colab's own port proxy

`google.colab.output.serve_kernel_port_as_window()` can expose a port, but the
proxy is built around Jupyter kernel channels and does not reliably carry the
long-lived bidirectional WebSocket the VS Code web client needs. The editor may
load and then lose its terminal. That is why every working setup uses a tunnel.

## Supervision

```bash
bash colab/supervise.sh
```

* restarts `serve.sh` with exponential backoff (2 s → 60 s) after a crash
* resets the backoff once the stack has been up for `CS_HEALTHY_AFTER` seconds
* shuts down after `CS_IDLE_TIMEOUT` seconds with zero established connections
* shuts down after `CS_MAX_RUNTIME` seconds, ahead of Colab's own hard cap

## Configuration

| Variable | Default | Meaning |
|---|---|---|
| `NGROK_TOKEN` | — | ngrok authtoken, or add it once with `ngrok config add-authtoken` |
| `NGROK_DOMAIN` | — | reserved domain → stable hostname, no rotation |
| `NGROK_BASIC_AUTH` | — | `user:pass`, rejects scanners at ngrok's edge |
| `CF_TOKEN` | — | Cloudflare named-tunnel token |
| `CF_TUNNEL_URL` | — | public hostname to report for a named tunnel |
| `CS_TUNNEL` | `auto` | `auto` \| `ngrok` \| `cloudflared` \| `none`. `auto` = ngrok when it has credentials, else a cloudflared quick tunnel (no account needed) |
| `CS_PORT` | auto | pin a port, or leave empty to scan `CS_PORT_RANGE` and take the first free one. The tunnel always maps to the chosen port |
| `CS_PORT_RANGE` | `3000-5000` | scan range when `CS_PORT` is empty. Avoids 8080, which `datalab/web/app.js` owns on some Colab runtimes. The choice is recorded in `$CS_STATE_DIR/port` |
| `CS_PASSWORD` | generated | fixed password; otherwise persisted to the state dir |
| `CS_STATE_DIR` | `/content/.code-server` | data, extensions, logs, pidfiles, password |
| `CS_CACHE_DIR` | `$CS_STATE_DIR/cache` | release tarball cache — point at Drive to skip re-downloads |
| `CS_VERSION` | latest | pin a code-server version |
| `CS_PREFIX` | `/usr/local` | install prefix |
| `CS_EXTENSIONS` | — | comma-separated extension IDs, installed once |
| `CS_IDLE_TIMEOUT` | `1800` | supervise: idle seconds before shutdown; `0` disables the idle watchdog |
| `CS_MAX_RUNTIME` | `39600` | supervise: total seconds before shutdown |
| `CS_MAX_RESTARTS` | `20` | supervise: consecutive failures before giving up |
| `CS_ALLOW_INSECURE` | `0` | must be `1` to accept `--no-auth` |
| `CS_STATE_DIR` on Drive | — | `CS_STATE_DIR=/content/drive/MyDrive/.cs` keeps extensions across runtimes at some FS latency cost |

## Diagnostics

```bash
bash colab/doctor.sh
```

Reports architecture, OS, memory, `/content` free space, code-server version,
tunnel credentials, live processes, port state, established connections,
`/healthz`, log sizes. Read-only. The password is hidden in its output; read
`$CS_STATE_DIR/credentials.txt` (mode 0600) if you need it.

## Why these flags

| Flag | Reason |
|---|---|
| `--bind-addr 127.0.0.1:PORT` | the tunnel agent is in the same VM; nothing needs a public bind |
| `--auth password` | tunnel hostnames get probed within minutes — never `--auth none` |
| `--disable-telemetry` / `--disable-update-check` | noise, plus writes to an ephemeral disk |
| `--disable-workspace-trust` | otherwise the trust modal eats the first click |
| `--user-data-dir` / `--extensions-dir` | keeps state in one persistable directory |
| `--trusted-origins '*'` | only if a proxy **rewrites** `Host` (`--host-header=rewrite`, Caddy `header_up Host`). That is the one case that looks like it needs a patch and does not. |

## Troubleshooting

| Symptom | Cause |
|---|---|
| blank page, no editor | ngrok warning page not clicked through |
| `ERR_NGROK_4018` | no authtoken: `ngrok config add-authtoken <token>` |
| `ERR_NGROK_108` | free tier allows one agent session; `bash colab/stop.sh` first |
| Cloudflare **error 1033** | the tunnel hostname resolves but no connector is attached — cloudflared died, or you are using a hostname from a previous run. Quick tunnels get a new hostname per start, so always take the URL from `credentials.txt`; `doctor.sh` now probes it and names 1033 |
| stale 404 URL | previous runtime's URL; re-run `serve.sh` |
| editor loads, terminal dies | WebSocket not carried by the path in use |
| `port 8080 is busy` | an orphan from an earlier run; `serve.sh` reclaims it automatically when it looks like ours. If it cannot, the pid and argv are printed — free it or use `--port=<n>` |
| restart loop, never reaching a URL | a previous supervisor is still alive relaunching the stack. `stop.sh` kills the supervisor *first*, then reclaims the port |
| stuck on `waiting for cloudflared URL` | quick-tunnel registration is slow or rate-limited; set `NGROK_TOKEN` |
| stuck on `waiting for .../healthz` | startup crash — read `code-server.log` |
| `extensions already installed` but missing | delete `$CS_STATE_DIR/.extensions-*` and restart |

## Hardening notes

* The password file is `0600`; `credentials.txt` is `0600` and holds the
  plaintext. For a stronger setup use code-server's `HASHED_PASSWORD`
  (argon2id) in `~/.config/code-server/config.yaml` and set `CS_PASSWORD` to
  nothing, accepting that the password is then not recoverable from disk.
* `NGROK_BASIC_AUTH` puts a second credential in front of code-server so port
  scanners never reach it. Cheap and worth it on a free hostname.
* Never combine a public tunnel with `--no-auth`, and never bind `0.0.0.0` on a
  host with a public interface.

## On the Colab terms

Colab's FAQ does not list remote-desktop or remote-IDE use as supported. This
works, but the runtime is ephemeral, the URL changes on restart unless you use a
named tunnel, and account-level throttling is always possible regardless of how
the process is named or wrapped.
''',
}

(BASE / "lib").mkdir(parents=True, exist_ok=True)
for rel, text in FILES.items():
    f = BASE / rel
    f.write_text(text)
    if f.suffix == ".sh":
        f.chmod(f.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

print(f"wrote {len(FILES)} files to {BASE}")
for rel in sorted(FILES):
    print(f"   {rel:16s} {(BASE / rel).stat().st_size:6d} bytes")

## 4. Install

Slow on purpose — this is where the 1–3 minutes go on a fresh runtime:
`apt-get` for the VS Code web client libraries, then the code-server tarball
(~100 MB), then both tunnel agents. Output streams live.

Re-running it is cheap: the tarball is cached in `CS_CACHE_DIR`, and apt is
skipped when the version already matches.

In [ ]:
import os, subprocess

env = dict(os.environ, WITH_CLOUDFLARED="1")  # cloudflared is the no-account fallback
subprocess.run(["bash", str(BASE / "install.sh")], check=True, env=env)

## 5. Start

Returns in about two seconds — the supervisor runs detached, so this cell does
not block and you are free to run the next one immediately.

In [ ]:
import os, signal, time, pathlib, subprocess

STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
STATE.mkdir(parents=True, exist_ok=True)

# Re-running must restart cleanly: retire the previous supervisor (it holds a
# lock), then let stop.sh release the port.
pid_file = STATE / "supervise.pid"
if pid_file.exists():
    try:
        pid = int(pid_file.read_text().strip())
        os.killpg(os.getpgid(pid), signal.SIGTERM)
        print("stopped previous supervisor", pid)
        for _ in range(20):
            time.sleep(0.5)
            os.kill(pid, 0)
    except (ValueError, ProcessLookupError, PermissionError):
        pass
    pid_file.unlink(missing_ok=True)
subprocess.run(["bash", str(BASE / "stop.sh")], check=False,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

creds = STATE / "credentials.txt"
creds.unlink(missing_ok=True)          # only accept a fresh write

log_path = STATE / "supervise.out"
fh = open(log_path, "w")
proc = subprocess.Popen(
    ["bash", str(BASE / "supervise.sh")],
    stdout=fh, stderr=subprocess.STDOUT,
    start_new_session=True,            # own process group, so stop can kill it cleanly
)
pid_file.write_text(str(proc.pid))

print(f"supervisor started (pid {proc.pid}), detached")
print(f"progress log : {log_path}")
print("next         : run the cell below for live progress and the URL")

## 6. Wait for the URL (live progress)

Streams the tail of `supervise.out` while the tunnel comes up, so a slow start is
visible instead of looking hung. Typical first run: health ~10 s, then 10–60 s
for the tunnel depending on provider.

In [ ]:
import os, time, pathlib
from IPython.display import display, clear_output, HTML

STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
creds = STATE / "credentials.txt"
log_path = STATE / "supervise.out"
pid_file = STATE / "supervise.pid"

BUDGET = 300
url = pw = None
reason = "timed out"

for elapsed in range(BUDGET):
    info = {}
    if creds.exists():
        raw = creds.read_text().strip()
        if "url=" in raw:
            info = dict(l.split("=", 1) for l in raw.splitlines() if "=" in l)
            if info.get("url"):
                url, pw, reason = info["url"], info.get("password"), "ok"
                break

    tail = ""
    if log_path.exists():
        tail = "\n".join(log_path.read_text().splitlines()[-10:])

    alive = True
    if pid_file.exists():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
        except (ValueError, ProcessLookupError, PermissionError):
            alive = False

    clear_output(wait=True)
    if not alive and not url:
        reason = "supervisor exited"
        break
    print(f"waiting {elapsed}s …  (log: {log_path})")
    print("─" * 60)
    print(tail or "(no output yet)")

    time.sleep(1)

clear_output(wait=True)
if url:
    display(HTML(
        f'<p style="font-size:17px;margin:8px 0"><b>code-server</b> &nbsp;'
        f'<a href="{url}" target="_blank">{url}</a></p>'
        f'<p style="margin:4px 0"><b>password</b> &nbsp;'
        f'<code style="font-size:15px">{pw}</code></p>'
    ))
    print(f"tunnel : {info.get('tunnel')}")
    print(f"state  : {STATE}")
    if info.get("tunnel") == "ngrok":
        print()
        print("ngrok free tier: click through the warning page before the editor loads.")
else:
    print(f"no URL yet ({reason}). Full progress log:\n")
    print(log_path.read_text() if log_path.exists() else "(no log)")
    print("\nNext: cell 8 (Status) for diagnostics, or cell 9 (Stop) then cell 6 again.")

## 7. Status

In [ ]:
import pathlib
STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
supervise_log = STATE / "supervise.out"

if supervise_log.exists():
    print("── supervisor ─────────────────────────────────────────────")
    print("\n".join(supervise_log.read_text().splitlines()[-12:]))
    print()

!bash "{BASE}/doctor.sh"

## 8. Stop

In [ ]:
import os, pathlib, signal

STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
pid_file = STATE / "supervise.pid"
if pid_file.exists():
    try:
        pid = int(pid_file.read_text().strip())
        os.killpg(os.getpgid(pid), signal.SIGTERM)   # supervisor was started detached
        print("stopped supervisor", pid)
    except (ValueError, ProcessLookupError, PermissionError):
        pass
    pid_file.unlink(missing_ok=True)

!bash "{BASE}/stop.sh"

## Troubleshooting

| Symptom | Cause / fix |
|---|---|
| cell 7 sits on "waiting for cloudflared URL" | Cloudflare quick-tunnel registration is slow or rate-limited; set `NGROK_TOKEN` for the faster path |
| stuck at `waiting for http://127.0.0.1:8080/healthz` | startup crash — read `code-server.log` |
| `port 8080 in use` | orphaned process: cell 9, then cell 6 |
| blank page after opening the URL | ngrok interstitial not clicked through, or set `TUNNEL = "cloudflared"` |
| URL 404s / changes | free ngrok rotates hostnames; set `NGROK_DOMAIN`, or use a named cloudflared tunnel |
| extensions missing | `rm -rf /content/.code-server/.extensions-*`, restart |

Useful from a shell cell:

```bash
tail -n 60 /content/.code-server/supervise.out
tail -n 60 /content/.code-server/code-server.log
bash /content/colab-cs/doctor.sh
```

The readable sources live in the repo at `colab/`; this notebook is generated
from them by `colab/build-notebook.py`, so prefer editing there.